# 02. Reconcile positions (Check-in 2 method gate)

This notebook answers the three Check-in 2 questions on the **full data**:

1. Does the rebuild run on the full data?
2. How many breaks, classified how?
3. What did we assume about REVERSED, and what else could it mean?

It also runs the price and ledger checks we chose, sorts corporate actions into supported and quarantined, traces one record hop by hop, and writes the defect register.

**How to run:** from the repo root, `uv run jupyter lab`, open this file, then *Run All*. Thresholds and treatment choices live in `config/reconciliation.toml`; change a value there and rerun. Results go to `outputs/02_*.csv`. Nothing in `data/` is changed. The plan behind every choice is `docs/notes/CHECKIN2_PLAN.md`.

**Reading the checks:** cells that start with `# CHECK` stop the notebook if something does not add up (for example, groups that do not sum to their denominator). Checks against notebook 01's numbers only stop the notebook when the thresholds are the same as notebook 01's; after a threshold change they report instead.

## 1. Setup and loading

Every table is read **as text** (DuckDB, `all_varchar=true`), then converted explicitly. Values that fail to convert are kept and counted, never dropped. The row ledger records rows in and out at every load, filter and join.

In [ ]:
import csv
import platform
import time
import tomllib
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
RUN_START = time.time()


def find_repo_root(start=None):
    """Walk up from `start` until we find the folder that holds manifest.csv and data/."""
    folder = Path(start or Path.cwd()).resolve()
    for candidate in [folder, *folder.parents]:
        if (candidate / 'manifest.csv').is_file() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError(f'No folder with manifest.csv and data/ above {folder}')


ROOT = find_repo_root()
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
import os
CONFIG_PATH = Path(os.environ.get('RECONCILIATION_CONFIG') or 'config/reconciliation.toml')
CONFIG_PATH = CONFIG_PATH if CONFIG_PATH.is_absolute() else ROOT / CONFIG_PATH   # relative paths: from the repo root
with CONFIG_PATH.open('rb') as stream:   # an alternative config can be passed for tests; the default is the repo's
    CONFIG = tomllib.load(stream)

AS_OF_DATE = CONFIG['snapshot']['as_of_date']
TOLERANCE = CONFIG['tolerances']['quantity_match_tolerance']
GROSS_TOLERANCE = CONFIG['tolerances']['gross_amount_tolerance']
POLICY = CONFIG['policies']

ROW_LEDGER = []   # one line per load, filter and join: rows in, rows out, why


def log_rows(step, rows_in, rows_out, why):
    ROW_LEDGER.append({'step': step, 'rows in': rows_in, 'rows out': rows_out, 'why': why})


print('Python', platform.python_version(), '| pandas', pd.__version__, '| duckdb', duckdb.__version__)
SHUFFLE_SEED = CONFIG['robustness']['shuffle_seed']
print('config:', CONFIG_PATH.relative_to(ROOT) if CONFIG_PATH.is_relative_to(ROOT) else CONFIG_PATH)
print('Snapshot', AS_OF_DATE, '| quantity tolerance', TOLERANCE, '| shuffle seed', SHUFFLE_SEED, '| policies', POLICY)

In [ ]:
with (ROOT / 'manifest.csv').open(newline='', encoding='utf-8') as stream:
    MANIFEST = {row['table']: row for row in csv.DictReader(stream)}

con = duckdb.connect()


def load_text(name):
    """Read one table with every column as text. A blank field stays '' (as in notebook 01), never NULL or NaN."""
    path = (ROOT / MANIFEST[name]['file']).as_posix()
    frame = con.sql(f"SELECT * FROM read_csv('{path}', all_varchar=true, header=true)").df()
    frame = frame.fillna('')
    log_rows(f'load {name}', int(MANIFEST[name]['rows']), len(frame), 'read as text; manifest rows in, loaded rows out')
    return frame


tables = {name: load_text(name) for name in MANIFEST}
if SHUFFLE_SEED:   # robustness test: every result must be the same whatever the row order
    tables = {name: frame.sample(frac=1, random_state=SHUFFLE_SEED).reset_index(drop=True) for name, frame in tables.items()}
    print(f'every table shuffled with seed {SHUFFLE_SEED}')
accounts, securities = tables['accounts'], tables['securities']
transactions, pipeline_events = tables['transactions'], tables['pipeline_events']
positions, comparison_positions = tables['positions'], tables['comparison_positions']
corporate_actions, prices = tables['corporate_actions'], tables['prices']
fx_rates, ledger_nav = tables['fx_rates'], tables['ledger_nav']
print(f'loaded {len(tables)} tables in {time.time() - RUN_START:.0f}s')

In [ ]:
# CHECK: every table loaded with the row and column counts in the jury's manifest.
load_check = pd.DataFrame([
    {'table': name,
     'manifest rows': int(MANIFEST[name]['rows']), 'loaded rows': len(tables[name]),
     'manifest columns': int(MANIFEST[name]['columns']), 'loaded columns': tables[name].shape[1]}
    for name in MANIFEST])
load_check['result'] = np.where((load_check['manifest rows'] == load_check['loaded rows'])
                                & (load_check['manifest columns'] == load_check['loaded columns']), 'PASS', 'FAIL')
assert set(MANIFEST) == set(tables), 'the manifest and the loaded tables name different tables'
assert (load_check['result'] == 'PASS').all(), load_check[load_check['result'] == 'FAIL']
load_check

## 2. The rebuild: raw benchmark first, then each justified step

We rebuild every account + security from zero (the Guide's shared convention): BUY adds, SELL subtracts. The **raw benchmark** uses every row as delivered. Each later step is one justified change, and the table shows its effect. The main chain is fixed in `config/reconciliation.toml` before looking at results:

| Step | What it does | Why |
|---|---|---|
| C0 | every row, raw signed quantity | the raw benchmark |
| C1 | drop the second copy of exact and date-format-only duplicates | a copy of the same record is not a second trade |
| C2 | keys touched by a **conflicting** duplicate pair: *not evaluable* | we cannot know which copy is right |
| C3 | drop CANCELLED rows | a cancelled trade is not a holding |
| C4 | drop REVERSED rows (**R1**: the original stands) | R1 = BOOKED-only, the Guide's shared benchmark |
| C5 | negative quantities kept **raw**; their keys are flagged later | never take absolute values silently |
| C6 | keys with a **blank** quantity: *not evaluable* | unknown never becomes zero |
| C7 | trades with trade date up to the snapshot | `positions` is a trade-date book |
| C7s | same, by scheduled settlement date | reported beside C7; a plan, not proof of settlement |

A key's group against a reported table: *matched*, *different*, *only in rebuild* (nets to zero, or nonzero), *only in reported table*, or *not evaluable*. Rate A = matched / (matched + different); rate B = matched / keys in union.

In [ ]:
def normalise_date(raw_dates):
    """Turn YYYY-MM-DD and DD/MM/YYYY text into YYYY-MM-DD text. Blanks stay blank; other values are returned unchanged."""
    iso = pd.to_datetime(raw_dates, format='%Y-%m-%d', errors='coerce')
    slash = pd.to_datetime(raw_dates, format='%d/%m/%Y', errors='coerce')
    return iso.fillna(slash).dt.strftime('%Y-%m-%d').fillna(raw_dates)


def normalise_timestamp(raw_timestamps):
    """Normalise the date part (first 10 characters) and keep the time part as it is."""
    return normalise_date(raw_timestamps.str.slice(0, 10)) + raw_timestamps.str.slice(10)


def to_day(raw_dates):
    """Text date in either format -> Timestamp; NaT where blank or unreadable (the raw text is kept in its own column)."""
    return pd.to_datetime(normalise_date(raw_dates), format='%Y-%m-%d', errors='coerce')


# Explicit conversions; the raw text columns stay next to the converted ones.
trades = transactions.copy()
trades['row_number'] = np.arange(len(trades))                     # lineage only, never a key
trades['trade_day'] = to_day(trades['trade_date'])
trades['settlement_day'] = to_day(trades['settlement_date'])
trades['quantity_number'] = pd.to_numeric(trades['quantity'], errors='coerce')
trades['quantity_blank'] = trades['quantity'] == ''
trades['price_number'] = pd.to_numeric(trades['price'], errors='coerce')
trades['gross_number'] = pd.to_numeric(trades['gross_amount'], errors='coerce')
direction = trades['transaction_type'].map({'BUY': 1, 'SELL': -1})   # missing for any other type
trades['signed_quantity'] = direction * trades['quantity_number']

# Repeated transaction_ids: each pair is an exact copy, a copy that differs only by date format, or a real conflict.
copies_per_id = trades['transaction_id'].value_counts()
assert copies_per_id.max() <= 2, 'an ID appears more than twice; the pair logic below compares only two copies'
data_columns = [column for column in transactions.columns if column != 'transaction_id']
repeated = trades[trades['transaction_id'].duplicated(keep=False)]
repeated_copy = repeated.groupby('transaction_id').cumcount()
copy_a = repeated[repeated_copy == 0].set_index('transaction_id')[data_columns]
copy_b = repeated[repeated_copy == 1].set_index('transaction_id')[data_columns].loc[copy_a.index]


def normalise_transaction_dates(frame):
    frame = frame.copy()
    frame['trade_date'] = normalise_date(frame['trade_date'])
    frame['settlement_date'] = normalise_date(frame['settlement_date'])
    frame['event_timestamp'] = normalise_timestamp(frame['event_timestamp'])
    return frame


raw_difference = (copy_a != copy_b).any(axis=1)
real_difference = (normalise_transaction_dates(copy_a) != normalise_transaction_dates(copy_b)).any(axis=1)
pair_type = pd.Series('exact copy', index=copy_a.index)
pair_type[raw_difference & ~real_difference] = 'differs only by date format'
pair_type[real_difference] = 'real conflict'
trades['pair_type'] = trades['transaction_id'].map(pair_type).fillna('not repeated')
trades['copy_number'] = trades.groupby('transaction_id').cumcount()   # 0 = first copy in file order

# REVERSED rows point to a BOOKED original; R3 needs to know which REVERSED row comes first per original.
is_reversed = trades['status'] == 'REVERSED'
trades['is_reversed_original'] = ((trades['status'] == 'BOOKED')
                                  & trades['transaction_id'].isin(trades.loc[is_reversed, 'related_transaction_id']))
trades['is_first_reversal'] = is_reversed & ~trades['related_transaction_id'].where(is_reversed).duplicated()

conversion_report = pd.Series({
    'rows': len(trades),
    'trade_date blank or unreadable': int(trades['trade_day'].isna().sum()),
    'settlement_date blank': int((trades['settlement_date'] == '').sum()),
    'settlement_date unreadable (not blank)': int((trades['settlement_day'].isna() & (trades['settlement_date'] != '')).sum()),
    'quantity blank': int(trades['quantity_blank'].sum()),
    'quantity unreadable (not blank)': int((trades['quantity_number'].isna() & ~trades['quantity_blank']).sum()),
    'quantity negative': int((trades['quantity_number'] < 0).sum()),
    'transaction_type not BUY or SELL': int(direction.isna().sum()),
    'price blank or unreadable': int(trades['price_number'].isna().sum()),
    'gross_amount blank or unreadable': int(trades['gross_number'].isna().sum()),
})
print('duplicate pairs:', pair_type.value_counts().to_dict())
conversion_report.to_frame('count')

In [ ]:
# The steps (ported from notebook 01 section 9). Each takes and returns trade rows.
def raw_rows(rows):
    """C0: every row, raw signed quantity, nothing marked."""
    return rows.assign(quantity_used=rows['signed_quantity'], mark_not_evaluable=False)

def drop_second_copies(rows, pair_types):
    return rows[~((rows['copy_number'] >= 1) & rows['pair_type'].isin(pair_types))]

def drop_exact_and_format_copies(rows):
    return drop_second_copies(rows, ['exact copy', 'differs only by date format'])

def conflicts_not_evaluable(rows):
    return rows.assign(mark_not_evaluable=rows['mark_not_evaluable'] | (rows['pair_type'] == 'real conflict'))

def conflicts_keep_first(rows):
    return drop_second_copies(rows, ['real conflict'])

def conflicts_keep_second(rows):
    return rows[~((rows['pair_type'] == 'real conflict') & (rows['copy_number'] == 0))]

def conflicts_drop_both(rows):
    return rows[rows['pair_type'] != 'real conflict']

def drop_cancelled(rows):
    return rows[rows['status'] != 'CANCELLED']

def reversed_R1(rows):
    """R1: a REVERSED row is a status record and does not count; the BOOKED original stands."""
    return rows[rows['status'] != 'REVERSED']

def reversed_R2(rows):
    """R2: neither the REVERSED rows nor their BOOKED original count (same quantity as R3, but the key can vanish)."""
    return rows[(rows['status'] != 'REVERSED') & ~rows['is_reversed_original']]

def reversed_R3(rows):
    """R3: the first REVERSED row per original offsets it once; the second one does not count."""
    rows = rows[(rows['status'] != 'REVERSED') | rows['is_first_reversal']]
    offset = rows['status'] == 'REVERSED'
    return rows.assign(quantity_used=rows['quantity_used'].where(~offset, -rows['quantity_used']))

def reversed_R4_evidence_only(rows):
    """R4: every REVERSED row offsets: a double offset. Evidence of how the book seems built, never a treatment."""
    offset = rows['status'] == 'REVERSED'
    return rows.assign(quantity_used=rows['quantity_used'].where(~offset, -rows['quantity_used']))

def negatives_raw(rows):
    return rows                                    # nothing changes; the keys get a sign_conflict flag in section 4

def negatives_not_evaluable(rows):
    return rows.assign(mark_not_evaluable=rows['mark_not_evaluable'] | (rows['quantity_number'] < 0))

def negatives_absolute_sensitivity(rows):
    """Sensitivity only: a negative raw quantity counted as its absolute value."""
    negative = rows['quantity_number'] < 0
    return rows.assign(quantity_used=rows['quantity_used'].where(~negative, -rows['quantity_used']))

def blanks_not_evaluable(rows):
    return rows                                    # a row without a number already marks its key (see rebuild)

def blanks_zero_sensitivity(rows):
    """Sensitivity only: a blank quantity counted as 0 units."""
    return rows.assign(quantity_used=rows['quantity_used'].where(~rows['quantity_blank'], 0.0))

CUTOFF = pd.Timestamp(AS_OF_DATE)

def trade_date_cutoff(rows):
    """Trades with trade_date up to the snapshot; an unreadable date marks its key."""
    unknown = rows['trade_day'].isna()
    rows = rows[unknown | (rows['trade_day'] <= CUTOFF)]
    return rows.assign(mark_not_evaluable=rows['mark_not_evaluable'] | rows['trade_day'].isna())

def settlement_date_cutoff(rows):
    """Trades with scheduled settlement_date up to the snapshot; a blank or unreadable date marks its key."""
    unknown = rows['settlement_day'].isna()
    rows = rows[unknown | (rows['settlement_day'] <= CUTOFF)]
    return rows.assign(mark_not_evaluable=rows['mark_not_evaluable'] | rows['settlement_day'].isna())


# Policy switches from the config. R4 and the two sensitivity-only treatments are refused as main choices.
CONFLICT_STEPS = {'not_evaluable': conflicts_not_evaluable, 'keep_first': conflicts_keep_first,
                  'keep_second': conflicts_keep_second, 'drop_both': conflicts_drop_both}
REVERSED_STEPS = {'R1': reversed_R1, 'R2': reversed_R2, 'R3': reversed_R3}
NEGATIVE_STEPS = {'raw': negatives_raw, 'not_evaluable': negatives_not_evaluable}
BLANK_STEPS = {'not_evaluable': blanks_not_evaluable}
DATE_STEPS = {'trade': trade_date_cutoff, 'settlement': settlement_date_cutoff}
for setting, allowed in [('conflicting_pairs', CONFLICT_STEPS), ('reversed_main', REVERSED_STEPS),
                         ('reversed_side', REVERSED_STEPS), ('negatives', NEGATIVE_STEPS),
                         ('blanks', BLANK_STEPS), ('holding_date', DATE_STEPS)]:
    if POLICY[setting] not in allowed:
        raise ValueError(f'{setting} = {POLICY[setting]!r} is not allowed here; choose one of {sorted(allowed)}')
OTHER_DATE = 'settlement' if POLICY['holding_date'] == 'trade' else 'trade'

CHAIN = [  # (code, what the step does, function), fixed before looking at results
    ('C0', 'raw benchmark: every row', raw_rows),
    ('C1', 'drop exact and date-format-only copies', drop_exact_and_format_copies),
    ('C2', f"conflicting pairs: {POLICY['conflicting_pairs']}", CONFLICT_STEPS[POLICY['conflicting_pairs']]),
    ('C3', 'drop CANCELLED', drop_cancelled),
    ('C4', f"REVERSED: {POLICY['reversed_main']}", REVERSED_STEPS[POLICY['reversed_main']]),
    ('C5', f"negative quantities: {POLICY['negatives']}", NEGATIVE_STEPS[POLICY['negatives']]),
    ('C6', f"blank quantities: {POLICY['blanks']}", BLANK_STEPS[POLICY['blanks']]),
    ('C7', f"{POLICY['holding_date']}-date cut-off", DATE_STEPS[POLICY['holding_date']]),
]
OTHER_DATE_CODE = 'C7s' if OTHER_DATE == 'settlement' else 'C7t'
MAIN_STEPS = [function for _, _, function in CHAIN]
print('main chain:', ' -> '.join(f'{code} {what}' for code, what, _ in CHAIN))

In [ ]:
KEY = ['account_id', 'security_id']
POSITION_KEY = ['account_id', 'security_id', 'as_of_date']
GROUPS = ['matched', 'different', 'only in rebuild, nonzero', 'only in rebuild, nets to zero',
          'only in reported table', 'not evaluable (in both)', 'not evaluable (only in rebuild)']

REPORTED = {}
for reported_name, table in [('positions', positions), ('comparison_positions', comparison_positions)]:
    side = table[POSITION_KEY].copy()
    side['reported_quantity'] = pd.to_numeric(table['quantity'], errors='coerce')
    if 'transaction_count' in table.columns:
        side['transaction_count'] = pd.to_numeric(table['transaction_count'], errors='coerce')
    REPORTED[reported_name] = side


def apply_steps(rows, steps):
    for step in steps:
        rows = step(rows)
    return rows


def rebuild(rows):
    """One row per key: summed quantity, contributing rows, and whether the key is evaluable."""
    rows = rows.assign(no_number=rows['quantity_used'].isna())
    rebuilt = rows.groupby(KEY, as_index=False).agg(
        rebuilt_quantity=('quantity_used', 'sum'), contributing_rows=('quantity_used', 'size'),
        rows_without_number=('no_number', 'sum'), marked=('mark_not_evaluable', 'any'))
    rebuilt['not_evaluable'] = (rebuilt['rows_without_number'] > 0) | rebuilt['marked']
    return rebuilt


def compare_groups(rebuilt, reported):
    """Full outer comparison of a rebuild with one reported table; every key gets exactly one group."""
    merged = rebuilt.assign(as_of_date=AS_OF_DATE).merge(reported, on=POSITION_KEY, how='outer',
                                                         indicator=True, validate='one_to_one')
    in_both = merged['_merge'] == 'both'
    only_rebuild = merged['_merge'] == 'left_only'
    not_evaluable = merged['not_evaluable'].eq(True) | (in_both & merged['reported_quantity'].isna())
    merged['difference'] = merged['rebuilt_quantity'] - merged['reported_quantity']
    within = merged['difference'].abs() <= TOLERANCE
    nets_to_zero = merged['rebuilt_quantity'].abs() <= TOLERANCE
    group = pd.Series('', index=merged.index)
    group[merged['_merge'] == 'right_only'] = 'only in reported table'
    group[only_rebuild & not_evaluable] = 'not evaluable (only in rebuild)'
    group[only_rebuild & ~not_evaluable & nets_to_zero] = 'only in rebuild, nets to zero'
    group[only_rebuild & ~not_evaluable & ~nets_to_zero] = 'only in rebuild, nonzero'
    group[in_both & not_evaluable] = 'not evaluable (in both)'
    group[in_both & ~not_evaluable & within] = 'matched'
    group[in_both & ~not_evaluable & ~within] = 'different'
    assert (group != '').all(), 'a key was left without a group'
    merged['group'] = group
    return merged


def summarise(merged):
    counts = merged['group'].value_counts().reindex(GROUPS, fill_value=0)
    assert counts.sum() == len(merged), 'groups do not add up to the keys in the union'
    summary = {group: int(counts[group]) for group in GROUPS}
    summary['keys in union'] = len(merged)
    comparable = counts['matched'] + counts['different']
    summary['rate A: matched / (matched + different)'] = round(counts['matched'] / comparable, 6) if comparable else np.nan
    summary['rate B: matched / keys in union'] = round(counts['matched'] / len(merged), 6)
    return summary


def run_variant(code, variant, rows):
    """Rebuild from `rows` and compare with both reported tables: one result line per reported table."""
    rebuilt = rebuild(rows)
    lines = []
    for reported_name, reported in REPORTED.items():
        lines.append({'code': code, 'variant': variant, 'compared with': reported_name, 'rows used': len(rows),
                      'keys rebuilt': len(rebuilt), **summarise(compare_groups(rebuilt, reported))})
    return lines, rebuilt

In [ ]:
# The waterfall: C0 raw benchmark, the Guide's BOOKED-only benchmark, then each step of our chain.
waterfall_lines = []
lines, _ = run_variant('BOOKED-only', 'Guide shared benchmark: BOOKED rows only, nothing else changed',
                       apply_steps(trades, [raw_rows, drop_cancelled, reversed_R1]))
waterfall_lines += lines

rows = trades
for code, what, step in CHAIN:
    rows_in = len(rows)
    rows = step(rows)
    log_rows(f'chain {code}', rows_in, len(rows), what)
    lines, rebuilt = run_variant(code, what, rows)
    waterfall_lines += lines
MAIN_ROWS, MAIN_REBUILT = rows, rebuilt                       # the main chain, used by every later section

OTHER_ROWS = apply_steps(trades, MAIN_STEPS[:-1] + [DATE_STEPS[OTHER_DATE]])
lines, OTHER_REBUILT = run_variant(OTHER_DATE_CODE, f'{OTHER_DATE}-date cut-off (reported beside C7)', OTHER_ROWS)
waterfall_lines += lines

waterfall = pd.DataFrame(waterfall_lines)
waterfall.to_csv(OUTPUTS / '02_waterfall.csv', index=False)
waterfall[waterfall['compared with'] == 'positions'].set_index('code').drop(columns='compared with')

In [ ]:
# Sensitivities: the main chain with ONE step swapped for an alternative. Delta = change against the main chain.
def main_chain_with(code, step):
    steps = list(MAIN_STEPS)
    steps[[c for c, _, _ in CHAIN].index(code)] = step
    return apply_steps(trades, steps)


alternatives = [('C2', f'conflicting pairs: {name}', function) for name, function in CONFLICT_STEPS.items()
                if name != POLICY['conflicting_pairs']]
alternatives += [('C4', f'REVERSED: {name}', function) for name, function in REVERSED_STEPS.items()
                 if name != POLICY['reversed_main']]
alternatives += [('C4', 'REVERSED: R4 every REVERSED row offsets (double offset; evidence only)', reversed_R4_evidence_only),
                 ('C5', 'negatives: absolute value (sensitivity only)', negatives_absolute_sensitivity),
                 ('C6', 'blanks: zero (sensitivity only)', blanks_zero_sensitivity)]
alternatives += [('C5', f'negatives: {name}', function) for name, function in NEGATIVE_STEPS.items()
                 if name != POLICY['negatives']]

main_lines = waterfall[waterfall['code'] == 'C7'].set_index('compared with')
sensitivity_lines = []
for code, alternative, step in alternatives:
    lines, _ = run_variant(code, alternative, main_chain_with(code, step))
    for line in lines:
        main = main_lines.loc[line['compared with']]
        line['matched vs main chain'] = line['matched'] - main['matched']
        line['different vs main chain'] = line['different'] - main['different']
        line['not evaluable vs main chain'] = ((line['not evaluable (in both)'] + line['not evaluable (only in rebuild)'])
                                               - (main['not evaluable (in both)'] + main['not evaluable (only in rebuild)']))
    sensitivity_lines += lines
sensitivities = pd.DataFrame(sensitivity_lines).rename(columns={'code': 'step swapped', 'variant': 'alternative'})
sensitivities.to_csv(OUTPUTS / '02_sensitivities.csv', index=False)
sensitivities[['step swapped', 'alternative', 'compared with', 'matched', 'different', 'matched vs main chain',
               'different vs main chain', 'not evaluable vs main chain']]

In [ ]:
# CHECK: the chain reproduces notebook 01 (outputs/01_rebuild_steps.csv) when the settings are the same.
NB01_POLICY = {'conflicting_pairs': 'not_evaluable', 'reversed_main': 'R1', 'negatives': 'raw',
               'blanks': 'not_evaluable', 'holding_date': 'trade'}   # notebook 01's chain, section 9.8
NB01_NAMES = {'C0': 'C0 raw benchmark', 'BOOKED-only': 'c alone: BOOKED only (Guide shared benchmark)',
              'C1': 'C1 drop exact and format-only copies', 'C2': 'C2 conflicting keys not evaluable',
              'C3': 'C3 drop CANCELLED', 'C4': 'C4 drop REVERSED (R1, BOOKED-only)', 'C5': 'C5 negatives kept raw',
              'C6': 'C6 blanks not evaluable', 'C7': 'C7 trade-date cut-off', 'C7s': 'C7s settlement-date cut-off'}
COMPARED = ['rows used', 'keys rebuilt', 'matched', 'different', 'only in rebuild, nonzero',
            'only in rebuild, nets to zero', 'only in reported table', 'not evaluable (in both)',
            'not evaluable (only in rebuild)', 'keys in union']
with (ROOT / 'config' / 'exploration.toml').open('rb') as stream:
    NB01_CONFIG = tomllib.load(stream)
same_settings = (NB01_CONFIG['snapshot']['as_of_date'] == AS_OF_DATE
                 and NB01_CONFIG['reconstruction']['quantity_match_tolerance'] == TOLERANCE
                 and all(POLICY[k] == v for k, v in NB01_POLICY.items()))
nb01_file = OUTPUTS / '01_rebuild_steps.csv'
if not nb01_file.is_file():
    print('Not checked: run notebook 01 first to create outputs/01_rebuild_steps.csv.')
else:
    nb01 = pd.read_csv(nb01_file).set_index(['variant', 'compared with'])
    ours = waterfall.assign(variant=waterfall['code'].map(NB01_NAMES)).set_index(['variant', 'compared with'])
    reproduction = (ours[COMPARED] - nb01.loc[ours.index, COMPARED]).abs().sum(axis=1).rename('total abs difference')
    same_data = int(nb01.loc[('C0 raw benchmark', 'positions'), 'rows used']) == len(trades)   # not a test fixture
    print(f'settings equal to notebook 01: {same_settings}; same data: {same_data}; variants compared: '
          f'{len(reproduction)}; variants that differ: {int((reproduction > 0).sum())}')
    if same_settings and same_data:
        assert (reproduction == 0).all(), reproduction[reproduction > 0]

## 3. Book against custodian, on the full set of keys

`positions` (our book) and `comparison_positions` (the custodian) are joined with a **full outer join** on account + security + as-of date, after checking that each side is unique at that grain. A key on one side only is reported as missing on the other side, never as a zero holding. This comparison does not use the rebuild, so keys the rebuild cannot evaluate still get an answer.

In [ ]:
# Grain check first: each side must be unique on account + security + as-of date.
for name, side in REPORTED.items():
    repeated_keys = int(side.duplicated(POSITION_KEY).sum())
    unreadable = int(side['reported_quantity'].isna().sum())
    print(f'{name}: {len(side):,} rows | repeated keys: {repeated_keys} | quantity unreadable: {unreadable} | '
          f'as_of_date values: {sorted(side["as_of_date"].unique())}')
    assert repeated_keys == 0, f'{name} is not unique at its grain; the comparison below would multiply rows'

book = REPORTED['positions'].rename(columns={'reported_quantity': 'book_quantity',
                                             'transaction_count': 'book_transaction_count'})
custodian = REPORTED['comparison_positions'].rename(columns={'reported_quantity': 'custodian_quantity'})
book_vs_custodian = book.merge(custodian, on=POSITION_KEY, how='outer', indicator='side', validate='one_to_one')
book_vs_custodian['side'] = book_vs_custodian['side'].map(
    {'both': 'both', 'left_only': 'only in positions', 'right_only': 'only in comparison_positions'})
log_rows('join positions <-> comparison_positions', len(book) + len(custodian), len(book_vs_custodian),
         'full outer join on account + security + as_of_date; rows out = keys in the union')

both_sides = book_vs_custodian['side'] == 'both'
unreadable = book_vs_custodian['book_quantity'].isna() | book_vs_custodian['custodian_quantity'].isna()
book_vs_custodian['book_minus_custodian'] = book_vs_custodian['book_quantity'] - book_vs_custodian['custodian_quantity']
agree = book_vs_custodian['book_minus_custodian'].abs() <= TOLERANCE
book_vs_custodian['group'] = np.select(
    [~both_sides, both_sides & unreadable, both_sides & agree],
    [book_vs_custodian['side'], 'not evaluable (in both)', 'matched'], default='different')

BVC_GROUPS = ['matched', 'different', 'not evaluable (in both)', 'only in positions', 'only in comparison_positions']
bvc_summary = (book_vs_custodian['group'].value_counts().reindex(BVC_GROUPS, fill_value=0)
               .rename('keys').to_frame())
bvc_summary['share of union'] = (bvc_summary['keys'] / len(book_vs_custodian)).round(6)
comparable = bvc_summary.loc[['matched', 'different'], 'keys'].sum()
print(f"keys in union: {len(book_vs_custodian):,} | rate A (matched / comparable): "
      f"{bvc_summary.loc['matched', 'keys'] / comparable:.6f} | rate B (matched / union): "
      f"{bvc_summary.loc['matched', 'keys'] / len(book_vs_custodian):.6f}")
differing = book_vs_custodian.loc[book_vs_custodian['group'] == 'different', 'book_minus_custodian']
print('book - custodian on differing keys: min / median / max =',
      f'{differing.min():,.2f} / {differing.median():,.2f} / {differing.max():,.2f}' if len(differing) else 'none')

In [ ]:
# CHECK: the book-vs-custodian groups add up to the union, and the keys in both add up to matched + different + not evaluable.
bvc_counts = book_vs_custodian['group'].value_counts()
in_both = int((book_vs_custodian['side'] == 'both').sum())
assert int(bvc_counts.sum()) == len(book_vs_custodian), 'a key has no group'
assert (bvc_counts.get('matched', 0) + bvc_counts.get('different', 0)
        + bvc_counts.get('not evaluable (in both)', 0)) == in_both, 'keys in both do not add up'
assert len(book_vs_custodian) == len(REPORTED['positions']) + len(REPORTED['comparison_positions']) - in_both
cross_file = OUTPUTS / '01_cross_check.csv'
if cross_file.is_file() and same_data:
    nb01_cross = pd.read_csv(cross_file).set_index('check')['this notebook']
    expected = {'merge: in both': in_both,
                'merge: only in positions': int(bvc_counts.get('only in positions', 0)),
                'merge: only in comparison_positions': int(bvc_counts.get('only in comparison_positions', 0))}
    for check, ours in expected.items():
        assert int(float(nb01_cross[check])) == ours, f'{check}: notebook 01 {nb01_cross[check]}, here {ours}'
    print('join counts equal notebook 01:', expected)
bvc_summary

## 4. Three sources: who disagrees? (layer 1 of the break classification)

Each key now has up to three numbers: our **rebuild** (main chain), the **book** (`positions`) and the **custodian** (`comparison_positions`). Layer 1 gives every key exactly one pattern:

- all three agree, or one source is the odd one out (book, custodian or rebuild), or all three differ;
- a one-sided pattern when a source has no record (missing is reported as missing, never as zero);
- *not evaluable* when the rebuild cannot be trusted for that key (conflicting duplicate, blank quantity, or another unreadable input).

When the book and the custodian agree with each other and only the rebuild differs, the custodian is **not** independent evidence against the book: both may share the same convention, or our reading may be wrong. Those keys stay *unresolved*.

In [ ]:
# Why a rebuilt key is not evaluable, from the rows behind it (only causes the main chain actually marks).
row_causes = MAIN_ROWS.assign(
    conflict=(MAIN_ROWS['pair_type'] == 'real conflict') & (POLICY['conflicting_pairs'] == 'not_evaluable'),
    blank=MAIN_ROWS['quantity_blank'],
    negative=(MAIN_ROWS['quantity_number'] < 0) & (POLICY['negatives'] == 'not_evaluable'),
    no_number_other=MAIN_ROWS['quantity_used'].isna() & ~MAIN_ROWS['quantity_blank'],
    date_unknown=MAIN_ROWS[f"{POLICY['holding_date']}_day"].isna())
key_causes = row_causes.groupby(KEY, as_index=False)[['conflict', 'blank', 'negative', 'no_number_other',
                                                      'date_unknown']].any()
CAUSE_NAMES = {'conflict': 'conflicting duplicate', 'blank': 'blank quantity', 'negative': 'negative quantity',
               'no_number_other': 'unreadable quantity or type', 'date_unknown': 'unreadable date'}

three_way = (MAIN_REBUILT[KEY + ['rebuilt_quantity', 'contributing_rows', 'not_evaluable']]
             .assign(in_rebuild=True)
             .merge(book[KEY + ['book_quantity', 'book_transaction_count']].assign(in_book=True), on=KEY, how='outer')
             .merge(custodian[KEY + ['custodian_quantity']].assign(in_custodian=True), on=KEY, how='outer')
             .merge(key_causes, on=KEY, how='left'))
for column in ['in_rebuild', 'in_book', 'in_custodian', 'not_evaluable', *CAUSE_NAMES]:
    three_way[column] = three_way[column].astype('boolean').fillna(False).astype(bool)
log_rows('three-way union (rebuild, book, custodian)', len(MAIN_REBUILT) + len(book) + len(custodian), len(three_way),
         'full outer joins on account + security; rows out = keys in the union')

def layer1_patterns(frame):
    """One pattern per key from the three quantities, which sources have the key, and rebuild evaluability."""
    r, b, c = frame['rebuilt_quantity'], frame['book_quantity'], frame['custodian_quantity']
    R, B, C = frame['in_rebuild'], frame['in_book'], frame['in_custodian']
    evaluable = R & ~frame['not_evaluable']
    rb, rc, bc = (r - b).abs() <= TOLERANCE, (r - c).abs() <= TOLERANCE, (b - c).abs() <= TOLERANCE
    all_three = evaluable & B & C
    ne_reason = frame[list(CAUSE_NAMES)].apply(
        lambda row: ' + '.join(name for column, name in CAUSE_NAMES.items() if row[column]) or 'other', axis=1)
    pattern = np.select(
        [R & frame['not_evaluable'],
         all_three & rb & rc & bc, all_three & rc & ~rb, all_three & rb & ~rc, all_three & bc & ~rb & ~rc,
         all_three & ~rb & ~rc & ~bc, all_three,
         ~R & B & C, evaluable & B & ~C, evaluable & ~B & C,
         evaluable & ~B & ~C & (r.abs() <= TOLERANCE), evaluable & ~B & ~C, ~R & B & ~C, ~R & ~B & C],
        ['not evaluable: ' + ne_reason,
         'all three agree', 'book is the odd one out', 'custodian is the odd one out', 'rebuild is the odd one out',
         'all three differ', 'tolerance edge case',
         'book + custodian, no trades', 'rebuild + book, no custodian', 'rebuild + custodian, no book',
         'rebuild only, nets to zero', 'rebuild only, nonzero', 'book only', 'custodian only'],
        default='')
    two_present_agree = np.select([~R & B & C, evaluable & B & ~C, evaluable & ~B & C], [bc, rb, rc], default=False)
    return pattern, two_present_agree


three_way['pattern'], three_way['two_present_agree'] = layer1_patterns(three_way)

pattern_summary = three_way['pattern'].value_counts().rename('keys').to_frame()
pattern_summary['share of union'] = (pattern_summary['keys'] / len(three_way)).round(6)
pattern_summary['of which the two present agree'] = (three_way[three_way['pattern'].isin(
    ['book + custodian, no trades', 'rebuild + book, no custodian', 'rebuild + custodian, no book'])]
    .groupby('pattern')['two_present_agree'].sum().reindex(pattern_summary.index).astype('Int64'))

three_way_summary = pd.concat([
    bvc_summary.reset_index().rename(columns={'group': 'group or pattern'}).assign(comparison='book vs custodian'),
    pattern_summary.reset_index().rename(columns={'pattern': 'group or pattern'}).assign(
        comparison='rebuild vs book vs custodian')], ignore_index=True)
three_way_summary.to_csv(OUTPUTS / '02_three_way_summary.csv', index=False)

In [ ]:
# CHECK: every key in the union of the three sources has exactly one pattern.
rebuild_keys = set(map(tuple, MAIN_REBUILT[KEY].to_numpy()))
book_keys = set(map(tuple, book[KEY].to_numpy()))
custodian_keys = set(map(tuple, custodian[KEY].to_numpy()))
assert len(three_way) == len(rebuild_keys | book_keys | custodian_keys), 'the union is the wrong size'
assert not three_way.duplicated(KEY).any(), 'a key appears twice'
assert (three_way['pattern'] != '').all() and three_way['pattern'].notna().all(), 'a key has no pattern'
assert int(pattern_summary['keys'].sum()) == len(three_way), 'patterns do not add up to the union'
print(f'keys in the union of rebuild, book and custodian: {len(three_way):,}')
pattern_summary

## 5. Why do they disagree? (layer 2: cause rules, one main class per key)

For every key where the sources disagree, the cause rules from `config/reconciliation.toml` are tested **in order** on the difference between the odd source and the rebuild; the first rule that fits sets the key's main class. A key that no rule explains is **unexplained** and stays visible. Each rule is a reproducible test, not a judgment:

| Rule | The difference equals... |
|---|---|
| book_counts_duplicate_copies | the signed quantity of the duplicate copies our chain removed (the source counts both copies) |
| book_double_offsets_reversed | what an R4 rebuild adds: every REVERSED row subtracted (a double offset) |
| duplicates_and_reversed | both of the above together |
| custodian_settlement_date_basis | the custodian quantity equals our settlement-date rebuild |
| sign_flipped_position | minus twice the rebuild: the source holds the opposite sign to the trades' net (added after the first run, when it showed up in the unexplained group; written as a rule and counted like the others) |

Keys that agree, one-sided keys and not-evaluable keys keep their layer-1 pattern as their class. **Flags** sit on top and never change the class: `sign_conflict` (a negative-quantity BUY on the key, whose `gross_amount` fits the absolute value) and `rebook_old_gross` (a rebook whose `gross_amount` still fits the cancelled quantity).

In [ ]:
# Each cause rule compares with the main chain where ONE convention is switched on.
def chain_variant(drop=(), swap=None):
    """The main chain without the steps in `drop`, and with `swap` = (code, function) replacing one step."""
    steps = []
    for code, _, function in CHAIN:
        if code in drop:
            continue
        steps.append(swap[1] if swap and code == swap[0] else function)
    return rebuild(apply_steps(trades, steps))


def delta_against_main(alternative, name):
    """Per key: alternative rebuild minus main rebuild (missing in the alternative -> missing, never zero)."""
    merged = MAIN_REBUILT[KEY + ['rebuilt_quantity']].merge(
        alternative[KEY + ['rebuilt_quantity']], on=KEY, how='left', suffixes=('', '_alternative'))
    merged[name] = merged['rebuilt_quantity_alternative'] - merged['rebuilt_quantity']
    return merged[KEY + [name]]


SETTLEMENT_REBUILT = OTHER_REBUILT if OTHER_DATE == 'settlement' else MAIN_REBUILT
alternatives = [
    delta_against_main(chain_variant(drop=['C1']), 'duplicate_delta'),
    delta_against_main(chain_variant(swap=('C4', reversed_R4_evidence_only)), 'reversed_delta'),
    delta_against_main(chain_variant(drop=['C1'], swap=('C4', reversed_R4_evidence_only)), 'both_delta'),
    SETTLEMENT_REBUILT[KEY + ['rebuilt_quantity', 'not_evaluable']].rename(
        columns={'rebuilt_quantity': 'settlement_quantity', 'not_evaluable': 'settlement_not_evaluable'})]
for frame in alternatives:
    rows_in = len(three_way)
    three_way = three_way.drop(columns=[c for c in frame.columns if c not in KEY and c in three_way.columns]).merge(
        frame, on=KEY, how='left', validate='one_to_one')
    assert len(three_way) == rows_in, 'a cause-rule join changed the number of keys'


def fits(difference, explained):
    """The difference equals the explained amount, and that amount is not zero."""
    return ((difference - explained).abs() <= TOLERANCE) & (explained.abs() > TOLERANCE)


RULE_TESTS = {
    'book_counts_duplicate_copies': lambda diff, source: fits(diff, three_way['duplicate_delta']),
    'book_double_offsets_reversed': lambda diff, source: fits(diff, three_way['reversed_delta']),
    'duplicates_and_reversed': lambda diff, source: (fits(diff, three_way['both_delta'])
                                                     & (three_way['duplicate_delta'].abs() > TOLERANCE)
                                                     & (three_way['reversed_delta'].abs() > TOLERANCE)),
    'custodian_settlement_date_basis': lambda diff, source: (
        (source == 'custodian') & ~three_way['settlement_not_evaluable'].astype('boolean').fillna(True).astype(bool)
        & ((three_way['custodian_quantity'] - three_way['settlement_quantity']).abs() <= TOLERANCE)),
    'sign_flipped_position': lambda diff, source: fits(diff, -2 * three_way['rebuilt_quantity']),
}
CAUSE_LABELS = {'book_counts_duplicate_copies': 'counts duplicate copies',
                'book_double_offsets_reversed': 'double-offsets REVERSED',
                'duplicates_and_reversed': 'duplicate copies + REVERSED double offset',
                'custodian_settlement_date_basis': 'custodian on settlement-date basis',
                'sign_flipped_position': 'opposite sign to the trades',
                'unexplained': 'unexplained'}
unknown_rules = set(CONFIG['classification']['cause_rules']) - set(RULE_TESTS)
assert not unknown_rules, f'config names cause rules that do not exist: {unknown_rules}'

r, b, c = three_way['rebuilt_quantity'], three_way['book_quantity'], three_way['custodian_quantity']
ODD_SOURCES = {'rebuild is the odd one out': [('book', b - r)],          # book = custodian here
               'book is the odd one out': [('book', b - r)],
               'custodian is the odd one out': [('custodian', c - r)],
               'all three differ': [('book', b - r), ('custodian', c - r)]}
cause = pd.Series('n/a', index=three_way.index)
for pattern, sources in ODD_SOURCES.items():
    open_keys = three_way['pattern'] == pattern
    for rule in CONFIG['classification']['cause_rules']:          # first rule that fits wins
        hit = pd.Series(False, index=three_way.index)
        for source, difference in sources:
            hit = hit | RULE_TESTS[rule](difference, source)
        cause[open_keys & hit] = rule
        open_keys = open_keys & ~hit
    cause[open_keys] = 'unexplained'
three_way['cause'] = cause

BREAK_KIND = {'all three agree': 'no', 'rebuild only, nets to zero': 'cleared-flag candidate',
              'book + custodian, no trades': 'one-sided', 'rebuild + book, no custodian': 'one-sided',
              'rebuild + custodian, no book': 'one-sided', 'rebuild only, nonzero': 'one-sided',
              'book only': 'one-sided', 'custodian only': 'one-sided'}
three_way['is a break'] = np.where(three_way['pattern'].str.startswith('not evaluable'), 'not evaluable',
                                   three_way['pattern'].map(BREAK_KIND).fillna('yes'))
three_way['main_class'] = np.where(three_way['cause'] == 'n/a', three_way['pattern'],
                                   three_way['pattern'] + ': ' + three_way['cause'].map(CAUSE_LABELS))

In [ ]:
# Flags: they sit on top of the class and never change it.
negative_rows = MAIN_ROWS[MAIN_ROWS['quantity_number'] < 0]
negative_rows = negative_rows.assign(gross_fits_absolute=(
    negative_rows['gross_number'] - negative_rows['quantity_number'].abs() * negative_rows['price_number']).abs()
    <= GROSS_TOLERANCE)
sign_conflict = negative_rows.groupby(KEY, as_index=False).agg(sign_conflict_rows=('transaction_id', 'size'))
print(f'negative-quantity rows in the main chain: {len(negative_rows):,} | types: '
      f'{negative_rows["transaction_type"].value_counts().to_dict()} | gross_amount fits the absolute value: '
      f'{int(negative_rows["gross_fits_absolute"].sum()):,}')

cancelled = (trades[trades['status'] == 'CANCELLED'].drop_duplicates('transaction_id')
             .set_index('transaction_id')['quantity_number'])
rebooks = MAIN_ROWS[MAIN_ROWS['related_transaction_id'].isin(cancelled.index)].copy()
rebooks['cancelled_quantity'] = rebooks['related_transaction_id'].map(cancelled)
fits_old = ((rebooks['gross_number'] - rebooks['cancelled_quantity'].abs() * rebooks['price_number']).abs()
            <= GROSS_TOLERANCE)
fits_own = ((rebooks['gross_number'] - rebooks['quantity_number'].abs() * rebooks['price_number']).abs()
            <= GROSS_TOLERANCE)
rebook_old_gross = (rebooks[(rebooks['quantity_number'] != rebooks['cancelled_quantity']) & fits_old & ~fits_own]
                    .groupby(KEY, as_index=False).agg(rebook_old_gross_rows=('transaction_id', 'size')))
print(f'rebooks in the main chain: {len(rebooks):,} | with a new quantity: '
      f'{int((rebooks["quantity_number"] != rebooks["cancelled_quantity"]).sum()):,} | gross_amount fits the old '
      f'quantity, not its own: {int(rebook_old_gross["rebook_old_gross_rows"].sum()):,}')

for flags in [sign_conflict, rebook_old_gross]:
    three_way = three_way.drop(columns=[c for c in flags.columns if c not in KEY and c in three_way.columns]).merge(
        flags, on=KEY, how='left', validate='one_to_one')
three_way['sign_conflict'] = three_way['sign_conflict_rows'].fillna(0) > 0
three_way['rebook_old_gross'] = three_way['rebook_old_gross_rows'].fillna(0) > 0

class_summary = (three_way.groupby(['is a break', 'pattern', 'cause', 'main_class'], as_index=False)
                 .agg(keys=('account_id', 'size'), sign_conflict_keys=('sign_conflict', 'sum'),
                      rebook_old_gross_keys=('rebook_old_gross', 'sum'))
                 .sort_values('keys', ascending=False))
class_summary['share of union'] = (class_summary['keys'] / len(three_way)).round(6)
class_summary.to_csv(OUTPUTS / '02_break_classes.csv', index=False)

In [ ]:
# Side view: the keys with a REVERSED row, rebuilt under the other REVERSED reading (R3: one offset per original).
reversal_keys = trades.loc[trades['status'] == 'REVERSED', KEY].drop_duplicates()
if POLICY['reversed_side'] != POLICY['reversed_main']:
    side_rebuilt = chain_variant(swap=('C4', REVERSED_STEPS[POLICY['reversed_side']]))
    side = three_way.merge(reversal_keys, on=KEY).drop(columns=['rebuilt_quantity', 'not_evaluable', 'in_rebuild'])
    side = side.merge(side_rebuilt[KEY + ['rebuilt_quantity', 'not_evaluable']], on=KEY, how='left')
    side['in_rebuild'] = side['rebuilt_quantity'].notna() | side['not_evaluable'].notna()
    side['not_evaluable'] = side['not_evaluable'].astype('boolean').fillna(False).astype(bool)
    side['side_pattern'], _ = layer1_patterns(side)
    reversed_side_view = (side.groupby(['pattern', 'side_pattern']).size().rename('keys').reset_index()
                          .rename(columns={'pattern': f"main reading ({POLICY['reversed_main']})",
                                           'side_pattern': f"side reading ({POLICY['reversed_side']})"}))
    reversed_side_view.to_csv(OUTPUTS / '02_reversed_side_view.csv', index=False)
    print(f'keys with a REVERSED row: {len(reversal_keys):,}')
    display(reversed_side_view)

In [ ]:
# CHECK: one main class per key; the classes add up to the union; the unexplained count is shown.
assert three_way['main_class'].notna().all() and (three_way['main_class'] != '').all(), 'a key has no class'
assert int(class_summary['keys'].sum()) == len(three_way), 'classes do not add up to the union'
per_pattern_layer2 = class_summary.groupby('pattern')['keys'].sum()
per_pattern_layer1 = three_way['pattern'].value_counts()
assert set(per_pattern_layer2.index) == set(per_pattern_layer1.index) and \
    (per_pattern_layer2 == per_pattern_layer1.reindex(per_pattern_layer2.index)).all(), \
    'layer 2 moved a key out of its layer-1 pattern'
unexplained = int(class_summary.loc[class_summary['cause'] == 'unexplained', 'keys'].sum())
print(f'unexplained breaks: {unexplained:,} of {int(class_summary.loc[class_summary["is a break"] == "yes", "keys"].sum()):,} breaks')
class_summary

## 6. One evidenced trace: a negative-quantity BUY

The record is **chosen by a rule** from `config/reconciliation.toml`, never typed in: among keys flagged `sign_conflict` that are present and evaluable in all three sources and carry no other flag, take the first by account, security and transaction ID. We then follow it hop by hop (Working Guide §4) and mark each hop **observed** (read directly from the data) or **inferred** (our reasoning or an assumption).

In [ ]:
def booked_before_cutoff(booked, cutoff):
    """True when the booked timestamp is at or before the cutoff; both normalised to YYYY-MM-DD first."""
    booked_text, cutoff_text = normalise_timestamp(pd.Series([booked, cutoff]))
    return booked_text <= cutoff_text


def fund_net_assets(ledger, fund_id, as_of_date):
    """Net assets of one fund, summed over its share classes (never sum nav_per_share), and the rows used."""
    rows = ledger[(ledger['fund_id'] == fund_id) & (normalise_date(ledger['as_of_date']) == as_of_date)]
    return pd.to_numeric(rows['net_assets'], errors='coerce').sum(), len(rows)


TRACE = CONFIG['trace']
FLAG_ROWS = {'sign_conflict': negative_rows,                                      # the rows behind each flag
             'rebook_old_gross': rebooks[(rebooks['quantity_number'] != rebooks['cancelled_quantity'])
                                         & fits_old & ~fits_own]}
OTHER_FLAGS = [flag for flag in FLAG_ROWS if flag != TRACE['flag']]
state = ['in_rebuild', 'in_book', 'in_custodian', 'not_evaluable', 'sign_conflict', 'rebook_old_gross',
         'pattern', 'main_class', 'rebuilt_quantity', 'book_quantity', 'custodian_quantity']
flagged = FLAG_ROWS[TRACE['flag']][KEY + ['transaction_id']].merge(three_way[KEY + state], on=KEY)
rule = (flagged[TRACE['flag']] & flagged['in_rebuild'] & flagged['in_book'] & flagged['in_custodian']
        & ~flagged['not_evaluable'])
no_other_flag = ~flagged[OTHER_FLAGS].any(axis=1)
trace_candidates = flagged[rule & no_other_flag].sort_values(TRACE['sort_by']).reset_index(drop=True)
if trace_candidates.empty:
    print('No key meets the full rule; dropping the "no other flag" condition, as the plan allows.')
    trace_candidates = flagged[rule].sort_values(TRACE['sort_by']).reset_index(drop=True)
print(f"keys flagged {TRACE['flag']}: {flagged[KEY].drop_duplicates().shape[0]:,} | meeting the trace rule: "
      f"{trace_candidates[KEY].drop_duplicates().shape[0]:,}")
chosen = trace_candidates.iloc[0]
TXN_ID, ACCOUNT_ID, SECURITY_ID = chosen['transaction_id'], chosen['account_id'], chosen['security_id']

trace_lines = []
def add(hop, table, field, value, evidence, note=''):
    trace_lines.append({'hop': hop, 'table': table, 'field': field, 'value': value, 'evidence': evidence, 'note': note})

# Hop 1, source: the transaction row as delivered (every raw field), and what its own amount says.
source_rows = transactions[transactions['transaction_id'] == TXN_ID]
for copy_number, (_, row) in enumerate(source_rows.iterrows()):
    for field, value in row.items():
        add('source', 'transactions', field if copy_number == 0 else f'{field} (copy {copy_number + 1})', value, 'observed')
src = trades[trades['transaction_id'] == TXN_ID].iloc[0]
implied = src['gross_number'] / src['price_number']
add('source', 'transactions', 'gross_amount / price', round(implied, 6), 'observed',
    f"quantity is {src['quantity']} on a {src['transaction_type']}, but the amount implies +{implied:,.2f} units: "
    'the sign contradicts gross_amount, a different field of the same record')
add('source', 'transactions', 'where it entered', src['source_system'], 'inferred',
    'the sign is already wrong in the row as delivered by this source system; nothing downstream changes it')

# Hop 2, pipeline: the matching pipeline record, and whether negatives cluster in one source or batch.
pipe_rows = pipeline_events[pipeline_events['transaction_id'] == TXN_ID]
for field, value in pipe_rows.iloc[0].items():
    add('pipeline', 'pipeline_events', field, value, 'observed')
add('pipeline', 'pipeline_events', 'pipeline records for this transaction_id', len(pipe_rows), 'observed')
add('pipeline', 'pipeline_events', 'booked before cutoff',
    bool(booked_before_cutoff(pipe_rows.iloc[0]['booked_timestamp'], pipe_rows.iloc[0]['cutoff_timestamp'])), 'observed',
    'timestamps compared in one platform clock')
negatives_by_source = (negative_rows['source_system'].value_counts(normalize=True).round(4).to_dict())
all_by_source = trades['source_system'].value_counts(normalize=True).round(4).to_dict()
add('pipeline', 'transactions', 'share of negative quantities by source_system', negatives_by_source, 'observed',
    f'share of all rows by source_system: {all_by_source}')
same_batch = pipeline_events['batch_id'] == pipe_rows.iloc[0]['batch_id']
negatives_in_batch = int(pipeline_events.loc[same_batch, 'transaction_id'].isin(negative_rows['transaction_id']).sum())
add('pipeline', 'pipeline_events', 'negative-quantity records in the same batch', negatives_in_batch, 'observed',
    f'out of {int(same_batch.sum()):,} records in that batch')

# Hop 3, position: the book against our rebuild.
book_row = positions[(positions['account_id'] == ACCOUNT_ID) & (positions['security_id'] == SECURITY_ID)].iloc[0]
key_rows = MAIN_ROWS[(MAIN_ROWS['account_id'] == ACCOUNT_ID) & (MAIN_ROWS['security_id'] == SECURITY_ID)].sort_values(
    ['trade_day', 'transaction_id'])                                   # listed in date order, never file order
for field in ['position_id', 'quantity', 'transaction_count', 'as_of_date', 'source']:
    add('position', 'positions', field, book_row[field], 'observed')
add('position', 'rebuild (main chain)', 'rebuilt quantity', chosen['rebuilt_quantity'], 'observed',
    f"{len(key_rows)} contributing rows: " + ', '.join(f"{t} {q}" for t, q in zip(key_rows['transaction_type'], key_rows['quantity'])))
add('position', 'rebuild (main chain)', 'book equals the raw rebuild',
    bool(abs(chosen['book_quantity'] - chosen['rebuilt_quantity']) <= TOLERANCE), 'observed',
    'the book carries the negative quantity as delivered, so the reconciliation matches and hides the error')
corrected = chosen['rebuilt_quantity'] + 2 * abs(src['quantity_number'])
add('position', 'rebuild (absolute-value sensitivity)', 'quantity if the sign followed gross_amount', corrected, 'inferred',
    f"differs from the book by 2 x {abs(src['quantity_number']):,.0f} units")

# Hop 4, custodian: does the second source catch it?
custodian_row = comparison_positions[(comparison_positions['account_id'] == ACCOUNT_ID)
                                     & (comparison_positions['security_id'] == SECURITY_ID)].iloc[0]
for field in ['comparison_id', 'quantity', 'as_of_date', 'source']:
    add('custodian', 'comparison_positions', field, custodian_row[field], 'observed')
shares_error = bool(abs(chosen['custodian_quantity'] - chosen['book_quantity']) <= TOLERANCE)
add('custodian', 'comparison_positions', 'custodian equals the book', shares_error, 'observed',
    'the custodian shares the error, so a book-vs-custodian check cannot catch it' if shares_error
    else 'the custodian differs from the book: independent evidence on this key')

# Hop 5, downstream: the fund the account feeds, and an indicative exposure (labelled; not a loss).
account = accounts[accounts['account_id'] == ACCOUNT_ID].iloc[0]
for field in ['fund_id', 'share_class', 'account_type', 'base_currency', 'status']:
    add('downstream', 'accounts', field, account[field], 'observed')
fund_total, fund_rows = fund_net_assets(ledger_nav, account['fund_id'], AS_OF_DATE)
add('downstream', 'ledger_nav', 'fund net_assets (all share classes, USD)', round(fund_total, 2), 'observed',
    f"{fund_rows} share-class rows; data_dictionary: net_assets is USD for the exercise")
security = securities[securities['security_id'] == SECURITY_ID]
asset_class = security.iloc[0]['asset_class'] if len(security) else 'not in securities'
add('downstream', 'securities', 'asset_class', asset_class, 'observed')
quotes = prices[(prices['security_id'] == SECURITY_ID) & (normalise_date(prices['price_date']) == AS_OF_DATE)].copy()
quotes['price_number'] = pd.to_numeric(quotes['price'], errors='coerce')
quotes['priority'] = pd.to_numeric(quotes['vendor_priority'], errors='coerce')
quotes = quotes[quotes['price_number'] > 0].sort_values('priority')
units_misstated = 2 * abs(src['quantity_number'])
add('downstream', 'rebuild', 'units misstated in the book', units_misstated, 'inferred',
    'if the sign follows gross_amount: the book holds -X where +X is expected')
if len(quotes):
    quote = quotes.iloc[0]
    rate = fx_rates[(fx_rates['currency'] == quote['currency'])
                    & (normalise_date(fx_rates['rate_date']) == AS_OF_DATE)]['rate_to_usd']
    rate = float(rate.iloc[0]) if len(rate) else np.nan
    exposure_usd = units_misstated * quote['price_number'] * rate
    add('downstream', 'prices + fx_rates', 'price used', f"{quote['price']} {quote['currency']} ({quote['vendor']}, "
        f"priority {quote['vendor_priority']})", 'inferred', 'lowest vendor_priority with a positive price on the snapshot date')
    add('downstream', 'fx_rates', 'rate_to_usd used', rate, 'inferred', 'USD per one unit of the price currency')
    add('downstream', 'derived', 'indicative exposure (USD)', round(exposure_usd, 2), 'inferred',
        'exposure, not a loss: units misstated x price x FX; quote basis assumed per unit' +
        ('; BOND: percent-of-par basis not verified' if asset_class == 'BOND' else ''))
    add('downstream', 'derived', 'indicative exposure (bps of fund net assets)',
        round(exposure_usd / fund_total * 1e4, 4), 'inferred', 'one basis point = 0.01 percent')
else:
    add('downstream', 'prices', 'price used', 'no eligible quote on the snapshot date', 'observed',
        'exposure in USD cannot be stated; units misstated is the measure')

# The control that would have caught it.
add('control', 'docs/notes/RULES_DRAFT.md', 'TXN_FIELDS_VALID',
    f'quantity > 0 and |gross_amount - quantity x price| <= {GROSS_TOLERANCE}', 'inferred',
    'Trade check at capture; owner: Trade operations lead; on fail: reject')

trace_table = pd.DataFrame(trace_lines)
trace_table['value'] = trace_table['value'].astype(str)
trace_table.to_csv(OUTPUTS / '02_trace.csv', index=False)
print(f'traced key: account {ACCOUNT_ID}, security {SECURITY_ID}, transaction {TXN_ID} | class: {chosen["main_class"]}')
trace_table[~trace_table['field'].isin(transactions.columns) | (trace_table['hop'] != 'source')]

In [ ]:
# CHECK: the trace helpers on small made-up inputs. Timestamps may come in either date format.
assert booked_before_cutoff('10/06/2026 10:15:00', '2026-06-02 18:00:00') == False, 'mixed formats compared as text'
assert booked_before_cutoff('02/06/2026 10:15:00', '2026-06-02 18:00:00') == True
two_dates = pd.DataFrame({'fund_id': ['F1', 'F1', 'F1'], 'net_assets': ['100', '20', '50'],
                          'as_of_date': ['2026-07-10', '10/07/2026', '2026-07-09']})
assert fund_net_assets(two_dates, 'F1', '2026-07-10') == (120.0, 2), 'another date was added into the fund total'

In [ ]:
# CHECK: the traced record follows the selection rule, every hop is marked observed or inferred,
# and the notebook source holds no literal record IDs (they must come from the data, never from the code).
import json, re
chosen = trace_candidates.iloc[0]
assert chosen['sign_conflict'] and chosen['in_rebuild'] and chosen['in_book'] and chosen['in_custodian']
assert not chosen['not_evaluable'] and not chosen['rebook_old_gross']
assert tuple(chosen[TRACE['sort_by']]) == min(map(tuple, trace_candidates[TRACE['sort_by']].to_numpy()))
assert set(trace_table['evidence']) <= {'observed', 'inferred'} and trace_table['evidence'].notna().all()
assert {'source', 'pipeline', 'position', 'custodian', 'downstream'} <= set(trace_table['hop'])
notebook_file = ROOT / 'notebooks' / '02_reconcile.ipynb'
source_text = ''.join(''.join(cell['source']) for cell in json.loads(notebook_file.read_text())['cells'])
literal_ids = re.findall(r'\b(?:ACC|SEC|TXN|CA|FND|LDG)\d{4,}\b', source_text)
assert not literal_ids, f'literal record IDs in the notebook source: {sorted(set(literal_ids))}'
print('trace selected by rule; no literal record IDs in the notebook source')

## 7. Ledger arithmetic: does net_assets / shares_outstanding give nav_per_share?

Rule `NAV_ARITHMETIC` (RULES_DRAFT §7): fund + share class + date unique; shares_outstanding > 0; |net_assets / shares_outstanding − nav_per_share| ≤ the tolerance in the config (one unit of nav_per_share's stored 6th decimal). Missing, unreadable or zero inputs are **not evaluable** and reported apart. We never sum nav_per_share. Cash, accruals and fees are not supplied, so a residual is something to investigate, not a proven NAV error.

In [ ]:
NAV_TOLERANCE = CONFIG['nav']['tolerance']
nav_check = ledger_nav.copy()
for column in ['net_assets', 'shares_outstanding', 'nav_per_share']:
    nav_check[f'{column}_number'] = pd.to_numeric(nav_check[column], errors='coerce')
nav_check['as_of_day'] = to_day(nav_check['as_of_date'])
nav_check['repeated key'] = nav_check.duplicated(['fund_id', 'share_class', 'as_of_date'], keep=False)

missing = nav_check[['net_assets_number', 'shares_outstanding_number', 'nav_per_share_number']].isna().any(axis=1)
zero_shares = nav_check['shares_outstanding_number'] <= 0
nav_check['implied_nav_per_share'] = nav_check['net_assets_number'] / nav_check['shares_outstanding_number'].where(~zero_shares)
nav_check['residual'] = nav_check['implied_nav_per_share'] - nav_check['nav_per_share_number']
nav_check['residual (bps of nav_per_share)'] = nav_check['residual'] / nav_check['nav_per_share_number'] * 1e4
nav_check['result'] = np.select([missing | zero_shares, nav_check['residual'].abs() <= NAV_TOLERANCE],
                                ['not evaluable', 'pass'], default='fail')
nav_check['reason'] = np.select([missing, zero_shares, nav_check['as_of_day'].isna()],
                                ['blank or unreadable input', 'shares_outstanding <= 0', 'as_of_date unreadable'], default='')
nav_check.to_csv(OUTPUTS / '02_nav_check.csv', index=False)

nav_summary = pd.DataFrame({
    'rows': [len(nav_check)],
    'repeated fund + class + date': [int(nav_check['repeated key'].sum())],
    'pass': [int((nav_check['result'] == 'pass').sum())],
    'fail': [int((nav_check['result'] == 'fail').sum())],
    'not evaluable': [int((nav_check['result'] == 'not evaluable').sum())],
    'tolerance': [NAV_TOLERANCE]}).T.rename(columns={0: 'value'})
fails = nav_check[nav_check['result'] == 'fail']
if len(fails):
    print('residual on failing rows (bps of nav_per_share): min / median / max =',
          f"{fails['residual (bps of nav_per_share)'].min():.4f} / {fails['residual (bps of nav_per_share)'].median():.4f}"
          f" / {fails['residual (bps of nav_per_share)'].max():.4f}")
    display(fails.reindex(fails['residual'].abs().sort_values(ascending=False).index)
            [['ledger_id', 'fund_id', 'share_class', 'net_assets', 'shares_outstanding', 'nav_per_share',
              'implied_nav_per_share', 'residual', 'residual (bps of nav_per_share)']].head(5))
print('not evaluable by reason:', nav_check.loc[nav_check['result'] == 'not evaluable', 'reason'].value_counts().to_dict())

In [ ]:
# CHECK: every ledger row is pass, fail or not evaluable, and the three add up to the rows in ledger_nav.
nav_counts = nav_check['result'].value_counts()
assert set(nav_counts.index) <= {'pass', 'fail', 'not evaluable'}, nav_counts
assert int(nav_counts.sum()) == len(ledger_nav), 'ledger rows do not add up'
nav_summary

## 8. Price inputs: vendor spread and scale, for every held security on the snapshot date

Rule `PRC_VENDOR_DISPERSION` (RULES_DRAFT §6), run on every security held in the book. For each one, the valid quotes on the snapshot date (price > 0) across vendors give a spread = (max − min) / mean. A max/min ratio in the configured band around **100** is flagged as a **scale** issue (percent-of-par vs per-unit, or a unit error). It is never auto-corrected, because quote basis and quantity denomination are not explicit in the data. Securities with no quote or a single quote are reported apart (nothing to compare), and so are quotes in **different currencies** (they cannot be compared without a stated conversion). A lower `vendor_priority` is a preference, not proof of accuracy.

Proposed, not run for Check-in 2 (rule, threshold and denominator are in RULES_DRAFT §6): holding coverage with vendor fallback (`PRC_QUOTE_ELIGIBLE`), freshness of `price_timestamp` against price date and NAV cutoff (flat price runs treated as look-alikes), and FX validity (`FX_RATE_VALID`; the USD rate ≠ 1 is an open question for faculty).

In [ ]:
PRICE_CONFIG = CONFIG['prices']
quotes_all = prices.copy()
quotes_all['price_day'] = normalise_date(quotes_all['price_date'])
quotes_all['price_number'] = pd.to_numeric(quotes_all['price'], errors='coerce')
snapshot_quotes = quotes_all[quotes_all['price_day'] == AS_OF_DATE]
log_rows('prices on the snapshot date', len(quotes_all), len(snapshot_quotes), 'price_date (either format) = snapshot')
invalid = snapshot_quotes['price_number'].isna() | (snapshot_quotes['price_number'] <= 0)
repeated_quotes = int(snapshot_quotes.duplicated(['security_id', 'vendor']).sum())
print(f'snapshot quotes: {len(snapshot_quotes):,} | blank, unreadable or <= 0: {int(invalid.sum())} | '
      f'repeated security + vendor: {repeated_quotes}')
valid = snapshot_quotes[~invalid]

held = positions[['security_id']].drop_duplicates()
stats = valid.groupby('security_id').agg(quotes=('price_number', 'size'), vendors=('vendor', 'nunique'),
                                         currencies=('currency', lambda c: '/'.join(sorted(set(c)))),
                                         low=('price_number', 'min'), high=('price_number', 'max'),
                                         mean=('price_number', 'mean')).reset_index()
vendor_check = held.merge(stats, on='security_id', how='left').merge(
    securities[['security_id', 'asset_class', 'currency']].rename(columns={'currency': 'master_currency'}),
    on='security_id', how='left')
vendor_check['quotes'] = vendor_check['quotes'].fillna(0).astype(int)
vendor_check['spread'] = (vendor_check['high'] - vendor_check['low']) / vendor_check['mean']
vendor_check['max / min'] = vendor_check['high'] / vendor_check['low']
currency_conflict = vendor_check['currencies'].fillna('').str.contains('/')
scale = vendor_check['max / min'].between(PRICE_CONFIG['scale_ratio_low'], PRICE_CONFIG['scale_ratio_high'])
vendor_check['result'] = np.select(
    [vendor_check['quotes'] == 0, vendor_check['quotes'] == 1, currency_conflict, scale,
     vendor_check['spread'] > PRICE_CONFIG['spread_threshold']],
    ['no quote (not evaluable)', 'single quote (not evaluable)', 'currency conflict (not evaluable)',
     'scale flag (ratio near 100)', 'spread above threshold'], default='pass')
vendor_check['quote currency differs from master'] = (vendor_check['currencies'].notna()
                                                      & vendor_check['master_currency'].notna()
                                                      & (vendor_check['currencies'] != vendor_check['master_currency']))
vendor_check['not in securities'] = vendor_check['asset_class'].isna()
vendor_check.to_csv(OUTPUTS / '02_vendor_check.csv', index=False)

vendor_summary = vendor_check['result'].value_counts().rename('securities').to_frame()
vendor_summary['share of held'] = (vendor_summary['securities'] / len(held)).round(6)
vendor_summary['of which BOND'] = vendor_check[vendor_check['asset_class'] == 'BOND']['result'].value_counts().reindex(
    vendor_summary.index).fillna(0).astype(int)
vendor_summary['of which not in securities'] = vendor_check[vendor_check['not in securities']]['result'].value_counts().reindex(
    vendor_summary.index).fillna(0).astype(int)
print(f"held securities: {len(held):,} | thresholds: spread {PRICE_CONFIG['spread_threshold']}, scale band "
      f"[{PRICE_CONFIG['scale_ratio_low']}, {PRICE_CONFIG['scale_ratio_high']}] | quote currency differs from master: "
      f"{int(vendor_check['quote currency differs from master'].sum())}")
evaluated = vendor_check[vendor_check['spread'].notna() & ~currency_conflict]
print('spread on securities with 2+ quotes in one currency: median / 99th pct / max =',
      f"{evaluated['spread'].median():.5f} / {evaluated['spread'].quantile(.99):.5f} / {evaluated['spread'].max():.3f}")

In [ ]:
# CHECK: every held security has exactly one result, and the results add up to the held securities.
held_securities = positions['security_id'].drop_duplicates()
assert vendor_check['security_id'].is_unique, 'a security appears twice'
assert set(vendor_check['security_id']) == set(held_securities), 'held securities missing from the vendor check'
assert int(vendor_summary['securities'].sum()) == len(held_securities), 'results do not add up to held securities'
vendor_summary

## 9. Corporate actions: supported or quarantined?

`corporate_actions` lists issuer events, not which accounts received them. Before any event-adjusted scenario (that comes for the final, kept separate from the trade-only rebuild), each event is sorted by rule `CA_TERMS_COMPLETE` (RULES_DRAFT §4), using the required terms per type in the config:

- **Supported:** the type's required terms are present and valid, the source security is mastered, and for MERGER and SPINOFF the target exists **and is in `securities`**.
- **Quarantined:** anything required is missing, with a reason code and the contract that would release it. REDEMPTION is quarantined by rule: the Guide gives no ratio convention for extinguished units, and we do not invent one.
- **Flags (unresolved, not quarantine):** a term the type does not use (for example a cash_rate on a split), and an ex-date after the effective date (the date definitions are an open question for faculty).

**Candidate holders** = accounts holding the source security in our trade-date rebuild just before the ex-date. They are candidates, not proven recipients: the extract has no posting log.

In [ ]:
CA_TERMS = {k: v for k, v in CONFIG['corporate_actions'].items() if isinstance(v, dict)}
RELEASE = {  # the contract or evidence that would release each quarantine reason
    'TYPE_NOT_CONFIGURED': 'a rule for this action type in config/reconciliation.toml',
    'SOURCE_NOT_MASTERED': 'the source security set up in securities',
    'MISSING_RATIO': 'the ratio from the issuer notice',
    'MISSING_CASH_RATE': 'the cash rate from the issuer notice',
    'MISSING_TARGET_SECURITY_ID': 'the resulting security named on the event',
    'TARGET_NOT_MASTERED': 'the target security set up in securities (identifiers, currency, asset class)',
    'NO_RATIO_CONVENTION': 'a stated convention for units extinguished per source unit, and the cash treatment',
    'DATE_UNREADABLE': 'readable ex_date and effective_date',
}
mastered = set(securities['security_id'])
ca = corporate_actions.copy()
ca['ex_day'], ca['effective_day'] = to_day(ca['ex_date']), to_day(ca['effective_date'])
ca['ratio_number'] = pd.to_numeric(ca['ratio'], errors='coerce')
ca['cash_rate_number'] = pd.to_numeric(ca['cash_rate'], errors='coerce')
present = {'ratio': ca['ratio_number'] > 0, 'cash_rate': ca['cash_rate_number'] > 0,
           'target_security_id': ca['target_security_id'] != '',
           'extinguished_units_convention': pd.Series(False, index=ca.index)}   # no such field in the extract
ca['target mastered'] = ca['target_security_id'].isin(mastered)
ca['source mastered'] = ca['security_id'].isin(mastered)

reasons, flags = [], []
for i, event in ca.iterrows():
    terms = CA_TERMS.get(event['action_type'])
    found, flagged = [], []
    if terms is None:
        found.append('TYPE_NOT_CONFIGURED')
    else:
        for term in terms['required']:
            if not present[term][i]:
                found.append('NO_RATIO_CONVENTION' if term == 'extinguished_units_convention' else f'MISSING_{term.upper()}')
        if 'target_security_id' in terms['required'] and present['target_security_id'][i] and not event['target mastered']:
            found.append('TARGET_NOT_MASTERED')
        extra = [term for term in terms['unused'] if present[term][i]]
        if extra:
            flagged.append('EXTRA_TERMS: ' + ', '.join(extra))
    if not event['source mastered']:
        found.append('SOURCE_NOT_MASTERED')
    if pd.isna(event['ex_day']) or pd.isna(event['effective_day']):
        found.append('DATE_UNREADABLE')
    elif event['ex_day'] > event['effective_day']:
        flagged.append('EX_AFTER_EFFECTIVE')
    reasons.append(' | '.join(found))
    flags.append(' | '.join(flagged))
ca['reason codes'] = reasons
ca['unresolved flags'] = flags
ca['verdict'] = np.where(ca['reason codes'] == '', 'supported', 'quarantined')
ca['what would release it'] = ca['reason codes'].map(
    lambda codes: ' | '.join(RELEASE[c] for c in codes.split(' | ')) if codes else '')

# Candidate holders: accounts holding the source security in the trade-date rebuild just before the ex-date.
ca_rows = MAIN_ROWS[MAIN_ROWS['security_id'].isin(ca['security_id'])]
holders, holders_not_evaluable = [], []
for _, event in ca.iterrows():
    if pd.isna(event['ex_day']):
        holders.append(np.nan); holders_not_evaluable.append(np.nan); continue
    before = ca_rows[(ca_rows['security_id'] == event['security_id']) & (ca_rows['trade_day'] < event['ex_day'])]
    rebuilt_before = rebuild(before) if len(before) else pd.DataFrame(columns=['rebuilt_quantity', 'not_evaluable'])
    evaluable_before = rebuilt_before[~rebuilt_before['not_evaluable'].astype(bool)]
    holders.append(int((evaluable_before['rebuilt_quantity'].abs() > TOLERANCE).sum()))
    holders_not_evaluable.append(int(rebuilt_before['not_evaluable'].astype(bool).sum()))
ca['candidate holders'] = holders
ca['holder keys not evaluable'] = holders_not_evaluable
ca_readiness = ca[['action_id', 'action_type', 'security_id', 'target_security_id', 'ex_date', 'effective_date',
                   'ratio', 'cash_rate', 'source mastered', 'target mastered', 'verdict', 'reason codes',
                   'what would release it', 'unresolved flags', 'candidate holders', 'holder keys not evaluable']]
ca_readiness.to_csv(OUTPUTS / '02_ca_readiness.csv', index=False)

ca_summary = (ca_readiness.groupby(['action_type', 'verdict'])
              .agg(events=('action_id', 'size'), candidate_holders=('candidate holders', 'sum'),
                   with_unresolved_flags=('unresolved flags', lambda f: int((f != '').sum())))
              .reset_index())
reason_counts = ca_readiness.loc[ca_readiness['verdict'] == 'quarantined', 'reason codes'].str.split(' | ', regex=False).explode().value_counts()
flag_counts = ca_readiness['unresolved flags'].str.split(' | ', regex=False).explode().replace('', np.nan).dropna().value_counts()
print('quarantine reasons (an event can have several):', reason_counts.to_dict())
print('unresolved flags (an event can have several):', flag_counts.to_dict())

In [ ]:
# CHECK: one row per event; each is supported or quarantined with a reason; an event whose type needs a
# target that is missing or not in securities is never supported.
assert len(ca_readiness) == len(corporate_actions) and ca_readiness['action_id'].is_unique
assert set(ca_readiness['verdict']) <= {'supported', 'quarantined'}
assert (ca_readiness.loc[ca_readiness['verdict'] == 'quarantined', 'reason codes'] != '').all()
needs_target = ca_readiness['action_type'].map(lambda t: 'target_security_id' in CA_TERMS.get(t, {}).get('required', []))
assert (ca_readiness.loc[needs_target & ~ca_readiness['target mastered'], 'verdict'] == 'quarantined').all()
ca_summary

## 10. Defect register

One line per finding: plain name, evidence status (**confirmed**, **unresolved** or **cleared**), the rule that tests it (a rule id from `docs/notes/RULES_DRAFT.md`; rules added on 16 Oct where the draft had a gap are marked *added*), the population (affected, out of how many eligible, and how many could not be evaluated), where it entered, and 2 or 3 example IDs. The examples are **produced by the code** (the first by sort order), never typed in. Notebook 01's 27 findings are recomputed here from the raw tables; the Check-in 2 findings come from sections 4 to 9.

The full seven-question write-ups of the four classes we take forward live in `docs/notes/DEFECT_CATALOG.md`.

In [ ]:
def key_ids(frame):
    """account_id/security_id text for key-level examples."""
    return frame['account_id'].astype(str) + '/' + frame['security_id'].astype(str)


ID_UNIVERSE = {
    'transaction_id': set(transactions['transaction_id']),
    'account/security': set(key_ids(three_way)),
    'security_id': set(securities['security_id']) | set(positions['security_id']) | set(prices['security_id'])
                   | set(corporate_actions['target_security_id']),
    'account_id': set(positions['account_id']) | set(accounts['account_id']),
    'action_id': set(corporate_actions['action_id']),
    'ledger_id': set(ledger_nav['ledger_id']),
    'currency': set(prices['currency']) | set(fx_rates['currency']),
}
register_lines = []


def finding(name, status, rule, stage, unit, affected_ids, eligible, not_evaluable, id_kind, how, affected=None):
    ids = pd.Series(list(affected_ids), dtype=object).dropna().astype(str)
    register_lines.append({
        'finding': name, 'status': status, 'rule': rule, 'stage': stage, 'unit': unit,
        'affected': int(affected if affected is not None else len(ids)), 'eligible': int(eligible),
        'not evaluable': int(not_evaluable), 'examples': ', '.join(sorted(ids.unique())[:3]),
        'id kind': id_kind, 'how detected': how})


n_rows = len(trades)
pair = trades['pair_type']
# --- Trade capture and pipeline (transactions, pipeline_events)
finding('Repeated transaction_id: exact full-row copy', 'confirmed', 'TXN_DUPLICATE', 'pipeline', 'rows',
        trades.loc[pair == 'exact copy', 'transaction_id'], n_rows, 0, 'transaction_id',
        'transaction_id appears twice and all columns are equal', affected=(pair == 'exact copy').sum())
finding('Repeated transaction_id: copies differ only by date format', 'confirmed', 'TXN_DUPLICATE', 'pipeline', 'rows',
        trades.loc[pair == 'differs only by date format', 'transaction_id'], n_rows, 0, 'transaction_id',
        'copies equal after normalising both date formats', affected=(pair == 'differs only by date format').sum())
finding('Repeated transaction_id: copies conflict on a fact', 'unresolved', 'TXN_DUPLICATE', 'pipeline', 'rows',
        trades.loc[pair == 'real conflict', 'transaction_id'], n_rows, 0, 'transaction_id',
        'copies still differ after date normalisation; which copy is right is unknown', affected=(pair == 'real conflict').sum())
pipe_repeated = pipeline_events['transaction_id'].duplicated(keep=False)
finding('Repeated transaction_id in the pipeline', 'confirmed', 'TXN_PIPELINE_ONE_TO_ONE (added)', 'pipeline', 'rows',
        pipeline_events.loc[pipe_repeated, 'transaction_id'], len(pipeline_events), 0, 'transaction_id',
        'pipeline_events not one-to-one with transactions', affected=pipe_repeated.sum())
slash_trade_date = trades['trade_date'].str.fullmatch(r'\d{2}/\d{2}/\d{4}')
finding('trade_date written as DD/MM/YYYY', 'confirmed', 'TXN_FIELDS_VALID', 'trade capture', 'rows',
        trades.loc[slash_trade_date, 'transaction_id'], n_rows, 0, 'transaction_id',
        'format count; readable, logged as a conformity issue', affected=slash_trade_date.sum())
blank_settlement = trades['settlement_date'] == ''
finding('Blank settlement_date', 'confirmed', 'TXN_FIELDS_VALID', 'trade capture', 'rows',
        trades.loc[blank_settlement, 'transaction_id'], n_rows, 0, 'transaction_id', 'blank field',
        affected=blank_settlement.sum())
dates_known = trades['trade_day'].notna() & trades['settlement_day'].notna()
settles_before = dates_known & (trades['settlement_day'] < trades['trade_day'])
finding('settlement_date before trade_date', 'confirmed', 'TXN_FIELDS_VALID', 'trade capture', 'rows',
        trades.loc[settles_before, 'transaction_id'], dates_known.sum(), (~dates_known).sum(), 'transaction_id',
        'both dates normalised and compared', affected=settles_before.sum())
quantity_known = trades['quantity_number'].notna()
negative = trades['quantity_number'] < 0
finding('Negative trade quantity (all BUY; gross_amount fits the absolute value)', 'confirmed', 'TXN_FIELDS_VALID',
        'trade capture', 'rows', trades.loc[negative, 'transaction_id'], quantity_known.sum(), (~quantity_known).sum(),
        'transaction_id', 'quantity < 0; the book keeps the raw sign, so the reconciliation matches (trace in section 6)',
        affected=negative.sum())
finding('Blank trade quantity', 'confirmed', 'TXN_FIELDS_VALID', 'trade capture', 'rows',
        trades.loc[trades['quantity_blank'], 'transaction_id'], n_rows, 0, 'transaction_id',
        'blank field; the book appears to count it as zero (sensitivity in section 2)', affected=trades['quantity_blank'].sum())
linked_cancelled = (trades['status'] == 'CANCELLED') & (trades['related_transaction_id'] != '')
dangling = linked_cancelled & ~trades['related_transaction_id'].isin(transactions['transaction_id'])
finding('CANCELLED row linked to an ID that does not exist', 'confirmed', 'TXN_CORRECTION_LINK', 'trade capture', 'rows',
        trades.loc[dangling, 'transaction_id'], linked_cancelled.sum(), 0, 'transaction_id',
        'related_transaction_id not found in transactions', affected=dangling.sum())
reversed_rows = trades[trades['status'] == 'REVERSED']
per_original = reversed_rows['related_transaction_id'].map(reversed_rows['related_transaction_id'].value_counts())
finding('More than one REVERSED row per original', 'unresolved', 'TXN_ONE_REVERSAL_PER_ORIGINAL (added)', 'trade capture',
        'rows', reversed_rows.loc[per_original > 1, 'transaction_id'], len(reversed_rows), 0, 'transaction_id',
        'REVERSED rows per related_transaction_id; meaning of REVERSED is an open question',
        affected=(per_original > 1).sum())
all_rebooks = trades[(trades['status'] == 'BOOKED') & trades['related_transaction_id'].isin(cancelled.index)].copy()
all_rebooks['cancelled_quantity'] = all_rebooks['related_transaction_id'].map(cancelled)
new_quantity = all_rebooks['quantity_number'] != all_rebooks['cancelled_quantity']
old_fit = (all_rebooks['gross_number'] - all_rebooks['cancelled_quantity'].abs() * all_rebooks['price_number']).abs() <= GROSS_TOLERANCE
own_fit = (all_rebooks['gross_number'] - all_rebooks['quantity_number'].abs() * all_rebooks['price_number']).abs() <= GROSS_TOLERANCE
finding('Rebook changes the quantity but keeps the cancelled gross_amount', 'unresolved', 'TXN_FIELDS_VALID', 'trade capture',
        'rows', all_rebooks.loc[new_quantity & old_fit & ~own_fit, 'transaction_id'], new_quantity.sum(), 0, 'transaction_id',
        'BOOKED -> CANCELLED links with a new quantity: gross_amount fits the old quantity, not its own; which is right is unknown')

# --- Master data
missing_account = ~positions['account_id'].isin(accounts['account_id'])
finding('Position on an account_id missing from accounts', 'confirmed', 'POS_KEYS_MASTERED (added)', 'master data', 'rows',
        positions.loc[missing_account, 'account_id'], len(positions), 0, 'account_id',
        'positions.account_id not in accounts', affected=missing_account.sum())
missing_security = ~positions['security_id'].isin(securities['security_id'])
finding('Position on a security_id missing from securities', 'confirmed', 'POS_KEYS_MASTERED (added)', 'master data', 'rows',
        positions.loc[missing_security, 'security_id'], len(positions), 0, 'security_id',
        'positions.security_id not in securities', affected=missing_security.sum())

# --- Corporate actions
has_target = ca['target_security_id'] != ''
target_missing = has_target & ~ca['target mastered']
finding('Corporate-action target security missing from securities', 'confirmed', 'CA_TERMS_COMPLETE', 'corporate actions',
        'events', ca.loc[target_missing, 'action_id'], has_target.sum(), 0, 'action_id',
        'target_security_id not in securities; event quarantined (section 9)', affected=target_missing.sum())
is_merger = ca['action_type'] == 'MERGER'
finding('MERGER without a target_security_id', 'confirmed', 'CA_TERMS_COMPLETE', 'corporate actions', 'events',
        ca.loc[is_merger & ~has_target, 'action_id'], is_merger.sum(), 0, 'action_id',
        'blank target on a merger; event quarantined', affected=(is_merger & ~has_target).sum())
is_redemption = ca['action_type'] == 'REDEMPTION'
finding('REDEMPTION with no convention for extinguished units', 'unresolved', 'CA_TERMS_COMPLETE', 'corporate actions', 'events',
        ca.loc[is_redemption, 'action_id'], is_redemption.sum(), 0, 'action_id',
        'the Guide defines no ratio convention for redemptions; quarantined, not guessed')
extra_terms = ca['unresolved flags'].str.contains('EXTRA_TERMS')
finding('ratio or cash_rate filled for an event type that does not use it', 'unresolved', 'CA_TERMS_COMPLETE',
        'corporate actions', 'events', ca.loc[extra_terms, 'action_id'], len(ca), 0, 'action_id',
        'terms present per action_type compared with the config table', affected=extra_terms.sum())
dates_readable = ca['ex_day'].notna() & ca['effective_day'].notna()
ex_after = dates_readable & (ca['ex_day'] > ca['effective_day'])
finding('ex_date after effective_date', 'unresolved', 'CA_TERMS_COMPLETE', 'corporate actions', 'events',
        ca.loc[ex_after, 'action_id'], dates_readable.sum(), (~dates_readable).sum(), 'action_id',
        'date definitions to confirm with faculty', affected=ex_after.sum())

# --- Prices and FX
quote_priority = pd.to_numeric(quotes_all['vendor_priority'], errors='coerce')
best_priority = quote_priority.min()
by_day = quotes_all.assign(priority=quote_priority).groupby(['security_id', 'price_day'])['priority'].min()
no_preferred = by_day[by_day > best_priority].reset_index()
finding('Preferred vendor has no price on a date', 'confirmed', 'PRC_QUOTE_ELIGIBLE', 'pricing', 'security-dates',
        no_preferred['security_id'], len(by_day), 0, 'security_id',
        'security x date grid: the lowest vendor_priority is absent; fallback needed', affected=len(no_preferred))
unpriced = ~positions['security_id'].isin(prices['security_id'])
finding('Held security with no price at all', 'confirmed', 'PRC_QUOTE_ELIGIBLE', 'pricing', 'position rows',
        positions.loc[unpriced, 'security_id'], len(positions), 0, 'security_id',
        'positions.security_id absent from prices', affected=unpriced.sum())
no_fx = ~prices['currency'].isin(fx_rates['currency'])
finding('Price currency with no FX rate', 'confirmed', 'FX_RATE_COVERAGE (added)', 'pricing', 'price rows',
        prices.loc[no_fx, 'currency'], len(prices), 0, 'currency', 'price currency not in fx_rates', affected=no_fx.sum())
usd = fx_rates[fx_rates['currency'] == 'USD']
usd_not_one = pd.to_numeric(usd['rate_to_usd'], errors='coerce') != 1
finding('USD rate to USD is not 1', 'unresolved', 'FX_RATE_VALID', 'pricing', 'rate rows',
        usd.loc[usd_not_one, 'currency'], len(usd), 0, 'currency', 'error or convention: open question for faculty',
        affected=usd_not_one.sum())
for result, status in [('spread above threshold', 'unresolved'), ('scale flag (ratio near 100)', 'unresolved')]:
    hit = vendor_check['result'] == result
    evaluable_securities = ~vendor_check['result'].str.contains('not evaluable')
    finding(f'Vendor quotes on the snapshot date: {result}', status, 'PRC_VENDOR_DISPERSION', 'pricing', 'securities',
            vendor_check.loc[hit, 'security_id'], evaluable_securities.sum(), (~evaluable_securities).sum(), 'security_id',
            'section 8; quote basis not explicit, so never auto-corrected')

# --- Positions: reconciliation (sections 3 to 5)
book_only = book_vs_custodian['side'] == 'only in positions'
finding('Position in the book with no custodian record', 'unresolved', 'POS_CUSTODIAN_BREAK', 'positions', 'keys',
        key_ids(book_vs_custodian[book_only]), len(book_vs_custodian), 0, 'account/security',
        'full outer join book vs custodian; trades support the book on most (section 4): likely a custodian feed gap')
breaks = three_way['is a break'] == 'yes'
class_rules = [
    ('Book and custodian count duplicate copies as trades', 'counts duplicate copies'),
    ('Book and custodian subtract every REVERSED row (double offset)', 'double-offsets REVERSED'),
    ('Book and custodian hold the opposite sign to the trades', 'opposite sign to the trades')]
for name, label in class_rules:
    hit = breaks & three_way['main_class'].str.endswith(label)
    finding(name, 'unresolved', 'POS_REBUILD_MATCHES', 'positions', 'keys', key_ids(three_way[hit]),
            len(three_way), int(three_way['is a break'].eq('not evaluable').sum()), 'account/security',
            'cause rule in section 5; book = custodian, so the custodian cannot catch it')
for name, pattern in [('Custodian differs from both the book and the trades', 'custodian is the odd one out'),
                      ('Book and custodian agree on a quantity the trades do not explain', 'rebuild is the odd one out')]:
    hit = (three_way['pattern'] == pattern) & (three_way['cause'] == 'unexplained')
    finding(name, 'unresolved', 'POS_CUSTODIAN_BREAK' if 'Custodian' in name else 'POS_REBUILD_MATCHES', 'positions',
            'keys', key_ids(three_way[hit]), len(three_way), int(three_way['is a break'].eq('not evaluable').sum()),
            'account/security', 'layer-1 pattern with no cause rule that fits (section 5)')
no_trades = three_way['pattern'] == 'book + custodian, no trades'
finding('Position with no contributing trade', 'unresolved', 'POS_REBUILD_MATCHES', 'positions', 'keys',
        key_ids(three_way[no_trades]), len(three_way), 0, 'account/security',
        'no opening balance is provided; never back-solved from the closing position')
booked_only = compare_groups(rebuild(apply_steps(trades, [raw_rows, drop_cancelled, reversed_R1])), REPORTED['positions'])
booked_only_groups = booked_only['group'].value_counts()
finding("Rebuilt quantity differs from the book (Guide's BOOKED-only benchmark)", 'unresolved', 'POS_REBUILD_MATCHES',
        'positions', 'keys', key_ids(booked_only[booked_only['group'] == 'different']),
        booked_only_groups.get('matched', 0) + booked_only_groups.get('different', 0),
        booked_only_groups.get('not evaluable (in both)', 0) + booked_only_groups.get('not evaluable (only in rebuild)', 0),
        'account/security', 'waterfall in section 2; classified in sections 4 and 5')

# The cleared-flag candidate: rule written in the plan before looking. Independent evidence: the custodian's
# matching absence (another table) and whether the book ever stores a zero-quantity row (the book's own convention).
nets_to_zero = three_way['pattern'] == 'rebuild only, nets to zero'
book_zero_rows = int((REPORTED['positions']['reported_quantity'].abs() <= TOLERANCE).sum())
cleared = nets_to_zero & ~three_way['in_custodian'] & ~three_way['in_book']
finding('Closed position not reported (trades net to zero, absent from book and custodian)',
        'cleared' if book_zero_rows == 0 and cleared.sum() == nets_to_zero.sum() else 'unresolved',
        'POS_REBUILD_MATCHES', 'positions', 'keys', key_ids(three_way[nets_to_zero]), len(three_way), 0,
        'account/security', f'the custodian is also silent on all of them, and positions stores {book_zero_rows} zero-quantity '
        'rows: neither source reports flat positions. Would reopen: a zero row in either source, or a later non-zero record')

# --- NAV
finding('NAV arithmetic does not hold (net_assets / shares_outstanding vs nav_per_share)', 'confirmed', 'NAV_ARITHMETIC',
        'NAV', 'ledger rows', nav_check.loc[nav_check['result'] == 'fail', 'ledger_id'],
        int((nav_check['result'] != 'not evaluable').sum()), int((nav_check['result'] == 'not evaluable').sum()),
        'ledger_id', 'section 7; which input is wrong is unknown, and cash or accruals are not supplied')

register = pd.DataFrame(register_lines)
register.to_csv(OUTPUTS / '02_defect_register.csv', index=False)

In [ ]:
# CHECK: every finding has a status, a rule and coherent denominators, and its example IDs exist in the data.
assert set(register['status']) <= {'confirmed', 'unresolved', 'cleared'}, register['status'].unique()
assert (register['rule'] != '').all() and (register['stage'] != '').all()
assert (register['eligible'] > 0).all() and (register['affected'] <= register['eligible']).all()
assert (register['not evaluable'] >= 0).all()
assert ((register['affected'] == 0) | (register['examples'] != '')).all(), 'a finding with records has no example'
for _, line in register.iterrows():
    examples = [e for e in line['examples'].split(', ') if e]
    missing_ids = [e for e in examples if e not in ID_UNIVERSE[line['id kind']]]
    assert not missing_ids, f"{line['finding']}: examples not found in the data: {missing_ids}"
print(f"findings: {len(register)} | confirmed {int((register['status'] == 'confirmed').sum())}, unresolved "
      f"{int((register['status'] == 'unresolved').sum())}, cleared {int((register['status'] == 'cleared').sum())}")
register[['finding', 'status', 'rule', 'stage', 'affected', 'eligible', 'not evaluable', 'unit', 'examples']]

## 11. Run record

Runtime, environment and the row ledger, so another team can check what this run did. **Robustness:** set `shuffle_seed` in `config/reconciliation.toml` to any number and rerun: every table's rows are shuffled after loading, and every output must stay the same. The only exception is the keep-first and keep-second sensitivities, which depend on row order by definition.

In [ ]:
run_info = pd.DataFrame([
    ('runtime_seconds', round(time.time() - RUN_START, 1)),
    ('python', platform.python_version()), ('pandas', pd.__version__), ('numpy', np.__version__),
    ('duckdb', duckdb.__version__), ('machine', platform.machine()), ('system', f'{platform.system()} {platform.release()}'),
    ('processor', platform.processor() or 'not reported'), ('config', CONFIG_PATH.name), ('shuffle_seed', SHUFFLE_SEED),
    ('as_of_date', AS_OF_DATE), ('quantity_match_tolerance', TOLERANCE),
], columns=['item', 'value'])
run_info.to_csv(OUTPUTS / '02_run_info.csv', index=False)
row_ledger = pd.DataFrame(ROW_LEDGER)
row_ledger.to_csv(OUTPUTS / '02_row_ledger.csv', index=False)
print(f"outputs written: {sorted(p.name for p in OUTPUTS.glob('02_*.csv'))}")
display(run_info)
row_ledger